 # Youtube Chatbot
  
 A complete Youtube Chatbot using RAG model to answer queries related to the videos

0. Importing Libraries

In [37]:
from youtube_transcript_api import YouTubeTranscriptApi
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_pinecone import PineconeVectorStore
from langchain_google_genai import GoogleGenerativeAIEmbeddings,ChatGoogleGenerativeAI
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

Loading API keys and env

In [2]:
load_dotenv()

True

# 1. Indexing

A. Document Loading

In [11]:
video_id = "RcGyVTAoXEU" # Youtube video ID 
ytt_api = YouTubeTranscriptApi() # You have to instantiate the API! 
transcript_list = ytt_api.fetch(video_id)

transcript = " ".join(chunk.text for chunk in transcript_list)
print(transcript)

I have a confession to make. But first, I want you to make
a little confession to me. In the past year,
I want you to just raise your hand if you've experienced
relatively little stress. Anyone? How about a moderate amount of stress? Who has experienced a lot of stress? Yeah. Me too. But that is not my confession. My confession is this: I am a health psychologist, and my mission is to help people
be happier and healthier. But I fear that something
I've been teaching for the last 10 years
is doing more harm than good, and it has to do with stress. For years I've been telling people,
stress makes you sick. It increases the risk of everything
from the common cold to cardiovascular disease. Basically, I've turned stress
into the enemy. But I have changed my mind about stress, and today, I want to change yours. Let me start with the study
that made me rethink my whole approach to stress. This study tracked 30,000 adults
in the United States for eight years, and they started by asking people

B. Text Splitting

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
chunks = splitter.create_documents([transcript])

len(chunks)

15

C. Embedding Generation and Storing in Vector Store

In [14]:
embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001") 

In [ ]:
vector_store = PineconeVectorStore.from_documents(
    documents=chunks,
    embedding=embeddings,
    index_name="yt-chatbot-dataset"
)

# 2. Retrieval 

A. Make retriever

In [20]:
retriever = vector_store.as_retriever(search_type="similarity",search_kwargs = {'k':4})
retriever.invoke("what is talked about a confession")

[Document(id='394ac201-1f21-4844-b2c7-7a093b873833', metadata={}, page_content="I have a confession to make. But first, I want you to make\na little confession to me. In the past year,\nI want you to just raise your hand if you've experienced\nrelatively little stress. Anyone? How about a moderate amount of stress? Who has experienced a lot of stress? Yeah. Me too. But that is not my confession. My confession is this: I am a health psychologist, and my mission is to help people\nbe happier and healthier. But I fear that something\nI've been teaching for the last 10 years\nis doing more harm than good, and it has to do with stress. For years I've been telling people,\nstress makes you sick. It increases the risk of everything\nfrom the common cold to cardiovascular disease. Basically, I've turned stress\ninto the enemy. But I have changed my mind about stress, and today, I want to change yours. Let me start with the study\nthat made me rethink my whole approach to stress. This study tra

# 3 Augmentation

A. Preparing LLM model

In [27]:
llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")

B. Setting Question and retrieving relevant docs

In [28]:
ques = "How to deal with stress?"
retrieved_docs = retriever.invoke(ques)

retrieved_docs

[Document(id='674e6280-0986-4640-8d58-f5308bd6da2b', metadata={}, page_content="science of stress reveals, that how you think about stress matters. So my goal as a health\npsychologist has changed. I no longer want\nto get rid of your stress. I want to make you better at stress. And we just did a little intervention. If you raised your hand and said you'd had a lot of stress\nin the last year, we could have saved your life, because hopefully the next time\nyour heart is pounding from stress, you're going to remember this talk and you're going to think to yourself, this is my body helping me\nrise to this challenge. And when you view stress in that way, your body believes you, and your stress response\nbecomes healthier. Now I said I have over a decade\nof demonizing stress to redeem myself from, so we are going to do\none more intervention. I want to tell you about one of the most under-appreciated"),
 Document(id='394ac201-1f21-4844-b2c7-7a093b873833', metadata={}, page_content="I hav

C. Preparing context from retrieved docs

In [29]:
context_text = "\n".join(doc.page_content for doc in retrieved_docs)
context_text

"science of stress reveals, that how you think about stress matters. So my goal as a health\npsychologist has changed. I no longer want\nto get rid of your stress. I want to make you better at stress. And we just did a little intervention. If you raised your hand and said you'd had a lot of stress\nin the last year, we could have saved your life, because hopefully the next time\nyour heart is pounding from stress, you're going to remember this talk and you're going to think to yourself, this is my body helping me\nrise to this challenge. And when you view stress in that way, your body believes you, and your stress response\nbecomes healthier. Now I said I have over a decade\nof demonizing stress to redeem myself from, so we are going to do\none more intervention. I want to tell you about one of the most under-appreciated\nI have a confession to make. But first, I want you to make\na little confession to me. In the past year,\nI want you to just raise your hand if you've experienced\nre

D. Preparing Prompt

In [32]:
prompt = PromptTemplate(
    template="""
        You are a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the context is insufficient, just say you don't know.

      {context}
      {ques} """,
    input_variables = ['context','ques']
)

E. Adding context to prompt

In [ ]:
final_prompt = prompt.invoke({"context":context_text,"ques":ques})

final_prompt

StringPromptValue(text="\n        You are a helpful assistant.\n        Answer ONLY from the provided transcript context.\n        If the context is insufficient, just say you don't know.\n\n      science of stress reveals, that how you think about stress matters. So my goal as a health\npsychologist has changed. I no longer want\nto get rid of your stress. I want to make you better at stress. And we just did a little intervention. If you raised your hand and said you'd had a lot of stress\nin the last year, we could have saved your life, because hopefully the next time\nyour heart is pounding from stress, you're going to remember this talk and you're going to think to yourself, this is my body helping me\nrise to this challenge. And when you view stress in that way, your body believes you, and your stress response\nbecomes healthier. Now I said I have over a decade\nof demonizing stress to redeem myself from, so we are going to do\none more intervention. I want to tell you about one o

# 4 Generation

In [35]:
answer = llm.invoke(final_prompt)
print(answer.text)

Based on the transcript, you can deal with stress by changing how you think about it:

*   **Rethink your stress response as helpful:** Instead of viewing physical changes (like a pounding heart or fast breathing) as anxiety or signs that you aren't coping, view them as your body being energized and preparing you to meet a challenge.
*   **Trust your body:** View a pounding heart as "preparing you for action" and faster breathing as a way to get "more oxygen to your brain." When you view the stress response as helpful, your body believes you and the response becomes healthier.
*   **Chase meaning:** Rather than trying to avoid discomfort, make decisions based on what creates meaning in your life and trust yourself to handle the stress that follows.
*   **Don't try to get rid of it:** The goal is not to eliminate stress, but to "get better at stress."


# 5. Building Chain for complete automation

A. creating a function to automate doc retrieval and arrangement

In [38]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

B. Building a parallel chain to run simultaneous tasks

In [46]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'ques': RunnablePassthrough()
})

C. Creating Parser instance

In [47]:
parser = StrOutputParser()

D. piping Final Chain and calling it 

In [48]:
main_chain = parallel_chain | prompt | llm | parser

In [49]:
main_chain.invoke("summarize this video")

'Based on the transcript provided, the speaker, a health psychologist, summarizes a new perspective on stress:\n\n*   **Perception of Stress Matters:** While she previously taught that stress was the "enemy" that caused illness and cardiovascular disease, she now argues that how you think about stress changes your body\'s physical response to it.\n*   **Biological Response:** In a typical stress response, heart rate increases and blood vessels constrict. however, when people view their stress response as helpful, their blood vessels stay relaxed. This creates a much healthier cardiovascular profile similar to moments of joy or courage.\n*   **The Role of Human Connection:** The stress response includes the release of oxytocin, which is enhanced by social contact and support. Reaching out to others or helping others under stress releases more of this hormone, making the stress response healthier and helping the body recover faster.\n*   **Resilience:** The speaker concludes that human c